# Continuous Performance

> **Task name:** `Continuous Performance`

**Track: Attention** | Can the model sustain target detection over a long, monotonous sequence?

The CPT is a foundational sustained attention paradigm in clinical neuropsychology.
The model sees a long sequence of 200 short sentences and must classify each as
TARGET (matching the specified category) or NON-TARGET.

Key features:
- **Low target frequency** (~15%) — most items are non-targets, creating monotony
- **Lure items** — sentences from a category similar to the target that test discrimination
- **Position effects** — does detection decay over the sequence?

**Metrics:** Hit rate, false alarm rate, d' (sensitivity), commission errors on lures.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def parse_yes_no(response: str, count: int) -> list[str]:
    """Parse YES/NO responses from numbered list."""
    response = strip_thinking(response)
    lines = response.strip().split("\n")
    answers = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        cleaned = re.sub(r"^\d+[\.)\:\-]\s*", "", line).strip().upper()
        if "YES" in cleaned and "NO" not in cleaned:
            answers.append("YES")
        elif "NO" in cleaned:
            answers.append("NO")
        elif cleaned in ["Y", "N"]:
            answers.append("YES" if cleaned == "Y" else "NO")
        else:
            answers.append(cleaned[:3])  # Take first 3 chars as fallback
    while len(answers) < count:
        answers.append("")
    return answers[:count]

In [ ]:
# Categories: target, lure (similar), and multiple non-target categories

CATEGORIES = {
    "weather": [
        "A severe thunderstorm warning was issued for the central plains region.",
        "Temperatures dropped below freezing overnight as an arctic front moved south.",
        "The hurricane intensified to Category 4 with sustained winds of 150 mph.",
        "Dense fog reduced visibility to less than 50 meters across the valley.",
        "Rainfall totals reached 180 millimeters in the 24-hour monitoring period.",
        "The monsoon season arrived three weeks earlier than historical averages.",
        "Wind speeds at the mountain observatory exceeded 120 kilometers per hour.",
        "The drought index reached extreme levels across four southeastern provinces.",
        "A tornado touched down near the industrial district causing structural damage.",
        "Hailstones the size of golf balls dented vehicles across the parking complex.",
        "The blizzard deposited 45 centimeters of snow in under six hours.",
        "Lightning strikes caused three separate brush fires in the dry grasslands.",
        "The heat wave persisted for 14 consecutive days above 40 degrees Celsius.",
        "A cold snap destroyed 30 percent of the citrus crop in the lowland orchards.",
        "Coastal flooding from the storm surge inundated roads two kilometers inland.",
    ],
    "climate_science": [  # LURE category — similar to weather but scientific
        "Atmospheric CO2 concentrations reached 425 parts per million at the monitoring station.",
        "Ice core data from Antarctica reveals temperature fluctuations spanning 800,000 years.",
        "Global mean surface temperature anomaly for the decade exceeded 1.2 degrees Celsius.",
        "Ocean acidification has reduced carbonate saturation by 26 percent since preindustrial times.",
        "Satellite measurements show Arctic sea ice volume declining at 13 percent per decade.",
        "Paleoclimate proxies indicate the current rate of warming is unprecedented in 10,000 years.",
        "The thermohaline circulation has weakened by 15 percent according to buoy array data.",
        "Permafrost thaw is releasing methane at rates 40 percent above 2010 baselines.",
        "Albedo reduction from snowpack loss amplifies regional warming by 0.3 degrees per decade.",
        "Climate model projections suggest a 2.7 degree increase by 2100 under current policies.",
    ],
    "history": [
        "The Treaty of Tordesillas divided newly discovered lands between Portugal and Spain in 1494.",
        "The fall of Constantinople to Ottoman forces in 1453 reshaped Mediterranean trade routes.",
        "The Gutenberg press produced its first complete Bible in Mainz around 1455.",
        "Viking longships reached the coast of Newfoundland approximately 500 years before Columbus.",
        "The Qing Dynasty governed China from 1644 until the revolution of 1912.",
        "The construction of the Suez Canal reduced shipping distances between Europe and Asia by 43 percent.",
        "The Library of Alexandria was the largest repository of knowledge in the ancient world.",
        "Hannibal crossed the Alps with an army including 37 war elephants in 218 BCE.",
        "The Rosetta Stone was discovered by French soldiers in Egypt in 1799.",
        "The Silk Road connected Chang'an to Constantinople spanning approximately 6,400 kilometers.",
    ],
    "technology": [
        "The transistor count in modern processors exceeds 100 billion on a single chip.",
        "Fiber optic cables carry data at 99.7 percent of the speed of light through glass cores.",
        "Solid-state batteries achieve energy densities of 500 watt-hours per kilogram.",
        "The James Webb telescope's primary mirror spans 6.5 meters in diameter.",
        "Quantum computers maintain coherent qubit states for up to 1,200 microseconds.",
        "Lithographic fabrication processes now operate at 2 nanometer feature sizes.",
        "Autonomous vehicle sensors process 4.5 terabytes of data per hour of driving.",
        "The international space station orbits Earth at an altitude of 408 kilometers.",
        "Nuclear fusion reactors achieved net energy gain for the first time in 2022.",
        "Satellite internet constellations provide coverage to 99 percent of the Earth's surface.",
    ],
    "sports": [
        "The marathon world record was lowered to under two hours in an exhibition event.",
        "The Olympic swimming pool requires a minimum depth of 2 meters for competition events.",
        "The Tour de France covers approximately 3,500 kilometers over 21 stages.",
        "Cricket test matches can last up to five days with each team batting twice.",
        "The FIFA World Cup final attracted an estimated 1.5 billion television viewers.",
        "Professional tennis players generate serve speeds exceeding 250 kilometers per hour.",
        "The basketball court measures exactly 28.7 meters in length for international play.",
        "Formula One cars experience lateral forces exceeding 6g during high-speed cornering.",
        "Competitive weightlifters in the super heavyweight class lift over 470 kilograms combined.",
        "The Ironman triathlon consists of a 3.86 km swim, 180 km cycle, and 42.2 km run.",
    ],
    "biology": [
        "The human genome contains approximately 3.2 billion base pairs of DNA.",
        "Mitochondria generate 90 percent of cellular energy through oxidative phosphorylation.",
        "The average human body contains approximately 37.2 trillion cells.",
        "Photosynthesis converts 6 CO2 and 6 H2O molecules into one glucose molecule.",
        "The axon of a motor neuron can conduct signals at speeds up to 120 meters per second.",
        "Red blood cells circulate for approximately 120 days before being recycled by the spleen.",
        "The ribosome translates mRNA into protein at a rate of about 20 amino acids per second.",
        "CRISPR-Cas9 can target specific DNA sequences with an accuracy exceeding 99 percent.",
        "The basal metabolic rate accounts for 60 to 75 percent of daily energy expenditure.",
        "Antibodies can recognize over 10 billion distinct molecular structures through V(D)J recombination.",
    ],
}

# Target categories and their lures
TARGET_CONFIGS = [
    {"target": "weather", "lure": "climate_science", "non_targets": ["history", "technology", "sports", "biology"]},
    {"target": "technology", "lure": "biology", "non_targets": ["weather", "climate_science", "history", "sports"]},
]

print(f"Categories: {list(CATEGORIES.keys())}")
print(f"Sentences per category: {', '.join(f'{k}={len(v)}' for k, v in CATEGORIES.items())}")

In [ ]:
random.seed(321)
SEQUENCE_LENGTH = 200
TARGET_FREQ = 0.15  # 15% targets
LURE_FREQ = 0.05    # 5% lures

data = []
task_id = 0

for config in TARGET_CONFIGS:
    target_cat = config["target"]
    lure_cat = config["lure"]
    non_target_cats = config["non_targets"]

    for variant in ["with_lures", "without_lures"]:
        # Build sequence
        sequence = []
        labels = []  # "TARGET", "LURE", "NON_TARGET"
        
        n_targets = int(SEQUENCE_LENGTH * TARGET_FREQ)
        n_lures = int(SEQUENCE_LENGTH * LURE_FREQ) if variant == "with_lures" else 0
        n_non_targets = SEQUENCE_LENGTH - n_targets - n_lures
        
        # Create items
        items = []
        target_sentences = random.choices(CATEGORIES[target_cat], k=n_targets)
        for s in target_sentences:
            items.append((s, "TARGET"))
        
        if n_lures > 0:
            lure_sentences = random.choices(CATEGORIES[lure_cat], k=n_lures)
            for s in lure_sentences:
                items.append((s, "LURE"))
        
        # Non-targets from multiple categories
        all_non_target = []
        for cat in non_target_cats:
            all_non_target.extend(CATEGORIES[cat])
        nt_sentences = random.choices(all_non_target, k=n_non_targets)
        for s in nt_sentences:
            items.append((s, "NON_TARGET"))
        
        random.shuffle(items)
        
        sequence = [s for s, _ in items]
        labels = [l for _, l in items]
        expected_answers = ["YES" if l == "TARGET" else "NO" for l in labels]
        
        lure_note = ""
        if variant == "with_lures":
            lure_note = (
                f"\n\nIMPORTANT: Some sentences may be about {lure_cat.replace('_', ' ')} — "
                f"these are NOT {target_cat} sentences. Only mark actual {target_cat} sentences as YES."
            )
        
        prompt_lines = [f"{i+1}. {s}" for i, s in enumerate(sequence)]
        prompt = (
            f"You will see {SEQUENCE_LENGTH} numbered sentences. For EACH sentence, determine whether "
            f"it is about **{target_cat}** (weather events, conditions, forecasts).\n\n"
            f"Respond with YES or NO for each, numbered to match.{lure_note}\n\n"
            + "\n".join(prompt_lines)
            + "\n\nAnswers (YES or NO for each):"
        )
        
        data.append({
            "task_id": task_id,
            "prompt": prompt,
            "expected_answers": expected_answers,
            "labels": labels,
            "target_category": target_cat,
            "lure_category": lure_cat if variant == "with_lures" else "none",
            "variant": variant,
            "num_items": SEQUENCE_LENGTH,
            "n_targets": n_targets,
            "n_lures": n_lures,
        })
        task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} CPT sequences")
print(f"By variant: {dict(df_all['variant'].value_counts())}")
print(f"By target category: {dict(df_all['target_category'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="continuous_performance",
    description="Sustained target detection in a 200-item sequence — classic CPT paradigm for LLMs"
)
def continuous_performance(
    llm,
    prompt: str,
    expected_answers: str,
    num_items: int,
) -> tuple[int, int]:
    response = llm.prompt(prompt)
    expected = json.loads(expected_answers) if isinstance(expected_answers, str) else expected_answers
    n = int(num_items)
    parsed = parse_yes_no(response, n)
    
    correct = 0
    for exp, act in zip(expected, parsed):
        if exp.upper().strip() == act.upper().strip():
            correct += 1
    
    return (correct, n)

## Run Evaluation

Uses `kbench.llm` — the default model. Use Kaggle's **"Add Models"** button to run across multiple models.

In [ ]:
eval_df = df_all.copy()
eval_df["expected_answers"] = eval_df["expected_answers"].apply(json.dumps)
task_eval_df = eval_df[["prompt", "expected_answers", "num_items"]].copy()

print(f"Running {len(task_eval_df)} CPT sequences with kbench.llm...")
runs = continuous_performance.evaluate(
    llm=kbench.llm,
    evaluation_data=task_eval_df,
    n_jobs=2,
    timeout=300,
    max_attempts=2,
)
results = runs.as_dataframe()
results["accuracy"] = results["result"].apply(
    lambda r: r[0] / r[1] if isinstance(r, tuple) and r[1] > 0 else float(r) if not isinstance(r, tuple) else 0
)
results = results.merge(
    eval_df[["prompt", "target_category", "variant", "labels", "expected_answers", "n_targets", "n_lures"]],
    on="prompt", how="left"
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "matplotlib"], check=True)
from scipy.stats import norm
import matplotlib.pyplot as plt

def compute_dprime(hr: float, far: float) -> float:
    hr = np.clip(hr, 0.01, 0.99)
    far = np.clip(far, 0.01, 0.99)
    return float(norm.ppf(hr) - norm.ppf(far))

print("=== Overall Accuracy ===")
print(results.groupby(["target_category", "variant"])["accuracy"].mean().to_string(float_format="%.3f"))

# Detailed SDT analysis per sequence
print("\n=== SDT Analysis ===")
for _, row in results.iterrows():
    expected = json.loads(row["expected_answers"]) if isinstance(row["expected_answers"], str) else row["expected_answers"]
    labels = json.loads(row["labels"]) if isinstance(row["labels"], str) else row["labels"]
    parsed = parse_yes_no(str(row.get("response", "")), len(expected))
    
    hits = sum(1 for e, p in zip(expected, parsed) if e == "YES" and p == "YES")
    misses = sum(1 for e, p in zip(expected, parsed) if e == "YES" and p != "YES")
    fas = sum(1 for e, p, l in zip(expected, parsed, labels) if e == "NO" and p == "YES" and l != "LURE")
    crs = sum(1 for e, p, l in zip(expected, parsed, labels) if e == "NO" and p != "YES" and l != "LURE")
    lure_fas = sum(1 for e, p, l in zip(expected, parsed, labels) if l == "LURE" and p == "YES")
    lure_total = sum(1 for l in labels if l == "LURE")
    
    total_targets = hits + misses
    total_non_targets = fas + crs
    hr = hits / total_targets if total_targets > 0 else 0.5
    far = fas / total_non_targets if total_non_targets > 0 else 0.5
    dprime = compute_dprime(hr, far)
    lure_commission = lure_fas / lure_total if lure_total > 0 else 0
    
    print(f"  {row['target_category']} ({row['variant']}): "
          f"HR={hr:.2%}, FAR={far:.2%}, d'={dprime:.2f}, "
          f"Lure commission={lure_commission:.2%} ({lure_fas}/{lure_total})")

print("\nNote: Lure commission rate > baseline FAR indicates the model is confused by similar categories.")

In [ ]:
# Position-based analysis (if raw responses available)
fig, ax = plt.subplots(figsize=(10, 5))

# Show accuracy breakdown
categories = ["Overall", "Hit Rate", "Correct Rejection", "Lure Rejection"]
values = [results["accuracy"].mean()]

# Compute aggregate stats
all_hrs, all_crrs, all_lrrs = [], [], []
for _, row in results.iterrows():
    expected = json.loads(row["expected_answers"]) if isinstance(row["expected_answers"], str) else row["expected_answers"]
    labels = json.loads(row["labels"]) if isinstance(row["labels"], str) else row["labels"]
    parsed = parse_yes_no(str(row.get("response", "")), len(expected))
    
    targets = [(e, p) for e, p in zip(expected, parsed) if e == "YES"]
    non_targets = [(e, p, l) for e, p, l in zip(expected, parsed, labels) if e == "NO" and l != "LURE"]
    lures = [(e, p) for e, p, l in zip(expected, parsed, labels) if l == "LURE"]
    
    if targets:
        all_hrs.append(sum(1 for e, p in targets if p == "YES") / len(targets))
    if non_targets:
        all_crrs.append(sum(1 for e, p, l in non_targets if p != "YES") / len(non_targets))
    if lures:
        all_lrrs.append(sum(1 for e, p in lures if p != "YES") / len(lures))

values.extend([np.mean(all_hrs) if all_hrs else 0, 
               np.mean(all_crrs) if all_crrs else 0,
               np.mean(all_lrrs) if all_lrrs else 0])
colors = ["steelblue", "seagreen", "darkorange", "mediumpurple"]

bars = ax.bar(categories, values, color=colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.0%}", ha="center", fontsize=12, fontweight="bold")

ax.set_ylabel("Rate")
ax.set_title("Continuous Performance Test: Detection Performance")
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig("cpt_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to cpt_results.png")